In [1]:
print("SaaS Analytics Project Started!")

SaaS Analytics Project Started!


In [2]:
import duckdb
import pandas as pd

df = duckdb.query("""
    SELECT *
    FROM read_parquet('../dataset/invoices.parquet')
""").df()

df.head()

,invoice_id,subscription_id,issued_on,period_start,period_end,amount,status,paid_on
0,1,1685,2024-12-20,2024-12-20,2025-01-19,1728.0,paid,2025-01-03
1,2,3325,2025-04-30,2025-04-30,2025-05-30,1508.0,paid,2025-05-20
2,3,2780,2024-06-14,2024-06-14,2024-07-14,457.0,paid,2024-06-20
3,4,3512,2025-03-26,2025-03-26,2025-04-25,137.0,paid,2025-03-30
4,5,2917,2025-08-26,2025-08-26,2025-09-25,147.0,paid,2025-09-08


In [3]:

# 1. Total rows and columns
print("Dataset Shape:", df.shape)

# 2. Column names
print("\nColumn Names:")
print(df.columns.tolist())

# 3. Data types and non-null counts
print("\nDataset Information:")
df.info()


Dataset Shape: (46000, 8)

Column Names:
['invoice_id', 'subscription_id', 'issued_on', 'period_start', 'period_end', 'amount', 'status', 'paid_on']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   invoice_id       46000 non-null  int32         
 1   subscription_id  46000 non-null  int32         
 2   issued_on        46000 non-null  datetime64[us]
 3   period_start     46000 non-null  datetime64[us]
 4   period_end       46000 non-null  datetime64[us]
 5   amount           46000 non-null  float64       
 6   status           46000 non-null  str           
 7   paid_on          40486 non-null  datetime64[us]
dtypes: datetime64[us](4), float64(1), int32(2), str(1)
memory usage: 2.5 MB


In [4]:
# Check how many invoices belong to each status
print("Invoice Status Counts:")
print(df["status"].value_counts(dropna=False))

# Check missing paid_on dates for each invoice status
# True means paid_on is missing; False means paid_on is present
print("\nMissing paid_on vs Invoice Status:")
print(pd.crosstab(
    df["status"],
    df["paid_on"].isna(),
    margins=True
))


Invoice Status Counts:
status
paid        40486
failed       3210
refunded     1360
open          944
Name: count, dtype: int64

Missing paid_on vs Invoice Status:
paid_on   False  True    All
status                      
failed        0  3210   3210
open          0   944    944
paid      40486     0  40486
refunded      0  1360   1360
All       40486  5514  46000


In [5]:
# Check basic statistical summary of invoice amounts
print("Invoice Amount Summary:")
print(df["amount"].describe())

# Count invoices with zero or negative amounts
print("\nZero Amount Invoices:", (df["amount"] == 0).sum())
print("Negative Amount Invoices:", (df["amount"] < 0).sum())

# Display the smallest and largest invoice amounts
print("\nMinimum Amount:", df["amount"].min())
print("Maximum Amount:", df["amount"].max())


Invoice Amount Summary:
count    46000.000000
mean       742.634957
std       1251.759671
min          0.000000
25%         71.000000
50%        367.000000
75%       1244.000000
max      27420.000000
Name: amount, dtype: float64

Zero Amount Invoices: 8430
Negative Amount Invoices: 0

Minimum Amount: 0.0
Maximum Amount: 27420.0


In [6]:
# Filter invoices where the amount is zero
zero_amount_df = df[df["amount"] == 0].copy()

# Check how many zero-amount invoices belong to each status
print("Zero-Amount Invoices by Status:")
print(zero_amount_df["status"].value_counts(dropna=False))

# Calculate the percentage of zero-amount invoices
zero_count = len(zero_amount_df)
total_count = len(df)

zero_percentage = (zero_count / total_count) * 100

print(f"\nZero-Amount Invoices: {zero_count}")
print(f"Percentage of Total Invoices: {zero_percentage:.2f}%")

# Display a few examples for further investigation
print("\nSample Zero-Amount Invoices:")
display(zero_amount_df.head(10))


Zero-Amount Invoices by Status:
status
paid        7400
failed       586
refunded     255
open         189
Name: count, dtype: int64

Zero-Amount Invoices: 8430
Percentage of Total Invoices: 18.33%

Sample Zero-Amount Invoices:


,invoice_id,subscription_id,issued_on,period_start,period_end,amount,status,paid_on
6,7,2469,2025-11-16,2025-11-16,2025-12-16,0.0,refunded,NaT
11,12,2042,2025-11-22,2025-11-22,2025-12-22,0.0,paid,2025-11-26
12,13,1391,2024-04-02,2024-04-02,2024-05-02,0.0,paid,2024-04-09
18,19,1223,2024-11-18,2024-11-18,2024-12-18,0.0,paid,2024-11-25
22,23,131,2023-10-06,2023-10-06,2023-11-05,0.0,paid,2023-10-20
23,24,1287,2024-06-09,2024-06-09,2024-07-09,0.0,paid,2024-06-23
24,25,3613,2025-09-20,2025-09-20,2025-10-20,0.0,paid,2025-10-04
25,26,893,2024-07-05,2024-07-05,2024-08-04,0.0,failed,NaT
27,28,2510,2025-02-15,2025-02-15,2025-03-17,0.0,paid,2025-02-17
33,34,1622,2024-11-04,2024-11-04,2024-12-04,0.0,paid,2024-11-06


In [7]:
# Load subscriptions and plans data from Parquet files
subscriptions_df = duckdb.query("""
    SELECT *
    FROM read_parquet('../dataset/subscriptions.parquet')
""").df()

plans_df = duckdb.query("""
    SELECT *
    FROM read_parquet('../dataset/plans.parquet')
""").df()

# Display column names to understand the available fields
print("Subscriptions Columns:")
print(subscriptions_df.columns.tolist())

print("\nPlans Columns:")
print(plans_df.columns.tolist())

# Display a few sample records from each dataset
print("\nSubscriptions Sample:")
display(subscriptions_df.head())

print("\nPlans Sample:")
display(plans_df.head())


Subscriptions Columns:
['subscription_id', 'organization_id', 'plan_id', 'started_on', 'ended_on', 'seats', 'cancel_reason']

Plans Columns:
['plan_id', 'plan_name', 'tier', 'monthly_price', 'seat_price', 'billing_interval', 'is_active']

Subscriptions Sample:


,subscription_id,organization_id,plan_id,started_on,ended_on,seats,cancel_reason
0,1,357,4,2025-03-23,2025-04-23,28,switched_vendor
1,2,16,2,2024-12-29,2025-03-30,3,missing_features
2,3,2116,2,2025-12-19,2025-12-30,13,non_payment
3,4,1415,2,2023-04-20,2023-05-23,4,switched_vendor
4,5,1891,8,2025-01-06,2025-03-31,16,price



Plans Sample:


,plan_id,plan_name,tier,monthly_price,seat_price,billing_interval,is_active
0,1,Free,free,0.0,0.0,monthly,False
1,2,Starter Monthly,starter,29.0,6.0,monthly,True
2,3,Starter Annual,starter,24.0,5.0,annual,False
3,4,Team Monthly,team,99.0,12.0,monthly,True
4,5,Team Annual,team,82.0,10.0,annual,True


In [8]:
# Join zero-amount invoices with their subscription and plan details
zero_invoice_details = (
    zero_amount_df
    .merge(
        subscriptions_df[["subscription_id", "plan_id"]],
        on="subscription_id",
        how="left"
    )
    .merge(
        plans_df[["plan_id", "plan_name", "tier", "monthly_price", "seat_price"]],
        on="plan_id",
        how="left"
    )
)

# Count zero-amount invoices by plan
print("Zero-Amount Invoices by Plan:")
print(
    zero_invoice_details["plan_name"]
    .value_counts(dropna=False)
)

# Compare invoice amounts with plan pricing information
print("\nZero-Amount Invoices by Plan Tier:")
display(
    zero_invoice_details
    .groupby("tier", dropna=False)
    .agg(
        invoice_count=("invoice_id", "count"),
        monthly_price=("monthly_price", "first"),
        seat_price=("seat_price", "first")
    )
    .sort_values("invoice_count", ascending=False)
)


Zero-Amount Invoices by Plan:
plan_name
Free    8430
Name: count, dtype: int64

Zero-Amount Invoices by Plan Tier:


,invoice_count,monthly_price,seat_price
tier,,,
free,8430,0.0,0.0


In [9]:
# Load the remaining SaaS datasets from Parquet files
organizations_df = duckdb.query("""
    SELECT * FROM read_parquet('../dataset/organizations.parquet')
""").df()

users_df = duckdb.query("""
    SELECT * FROM read_parquet('../dataset/users.parquet')
""").df()

usage_events_df = duckdb.query("""
    SELECT * FROM read_parquet('../dataset/usage_events.parquet')
""").df()

# Collect all loaded datasets for a quick overview
datasets = {
    "Organizations": organizations_df,
    "Users": users_df,
    "Plans": plans_df,
    "Subscriptions": subscriptions_df,
    "Invoices": df,
    "Usage Events": usage_events_df
}

# Display row counts, column counts, and column names
for name, data in datasets.items():
    print(f"\n--- {name} ---")
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())



--- Organizations ---
Shape: (3000, 7)
Columns: ['organization_id', 'org_name', 'industry', 'country', 'employee_band', 'signed_up_on', 'acquisition_channel']

--- Users ---
Shape: (26000, 6)
Columns: ['user_id', 'organization_id', 'email', 'role', 'joined_on', 'deactivated_on']

--- Plans ---
Shape: (8, 7)
Columns: ['plan_id', 'plan_name', 'tier', 'monthly_price', 'seat_price', 'billing_interval', 'is_active']

--- Subscriptions ---
Shape: (4200, 7)
Columns: ['subscription_id', 'organization_id', 'plan_id', 'started_on', 'ended_on', 'seats', 'cancel_reason']

--- Invoices ---
Shape: (46000, 8)
Columns: ['invoice_id', 'subscription_id', 'issued_on', 'period_start', 'period_end', 'amount', 'status', 'paid_on']

--- Usage Events ---
Shape: (180000, 6)
Columns: ['event_id', 'organization_id', 'user_id', 'occurred_at', 'event_type', 'quantity']


In [10]:
# Audit all datasets for missing values and duplicate rows
for name, data in datasets.items():
    print(f"\n--- {name} ---")

    # Count missing values in each column
    missing = data.isna().sum()
    print("Missing values:")
    print(missing[missing > 0] if missing.sum() > 0 else "None")

    # Count fully duplicated rows
    print("Duplicate rows:", data.duplicated().sum())



--- Organizations ---
Missing values:
acquisition_channel    528
dtype: int64
Duplicate rows: 0

--- Users ---
Missing values:
deactivated_on    22437
dtype: int64
Duplicate rows: 0

--- Plans ---
Missing values:
None
Duplicate rows: 0

--- Subscriptions ---
Missing values:
ended_on         2010
cancel_reason    2010
dtype: int64
Duplicate rows: 0

--- Invoices ---
Missing values:
paid_on    5514
dtype: int64
Duplicate rows: 0

--- Usage Events ---
Missing values:
user_id    25221
dtype: int64
Duplicate rows: 0


In [11]:
# Check subscription end dates and cancellation reasons
print("Subscriptions Without End Date:")
print(subscriptions_df["ended_on"].isna().value_counts())

print("\nCancellation Reasons:")
print(subscriptions_df["cancel_reason"].value_counts(dropna=False))

# Summarize subscriptions with and without an end date
subscription_summary = (
    subscriptions_df
    .assign(
        has_end_date=subscriptions_df["ended_on"].notna()
    )
    .groupby("has_end_date")
    .agg(
        subscription_count=("subscription_id", "count"),
        organizations=("organization_id", "nunique")
    )
    .reset_index()
)

print("\nSubscription Summary:")
display(subscription_summary)


Subscriptions Without End Date:
ended_on
False    2190
True     2010
Name: count, dtype: int64

Cancellation Reasons:
cancel_reason
NaN                 2010
price                596
missing_features     519
no_longer_needed     421
switched_vendor      416
non_payment          238
Name: count, dtype: int64

Subscription Summary:


,has_end_date,subscription_count,organizations
0,False,2010,2010
1,True,2190,1795


In [12]:
# Create a first summary of subscription and invoice performance

# Count subscriptions with and without an end date
total_subscriptions = len(subscriptions_df)
without_end_date = subscriptions_df["ended_on"].isna().sum()
with_end_date = subscriptions_df["ended_on"].notna().sum()

# Calculate invoice and payment metrics
total_invoices = len(df)
paid_invoices = (df["status"] == "paid").sum()
failed_invoices = (df["status"] == "failed").sum()
refunded_invoices = (df["status"] == "refunded").sum()
open_invoices = (df["status"] == "open").sum()

# Sum invoice amounts by status for an initial revenue review
invoice_amount_by_status = (
    df.groupby("status", dropna=False)
    .agg(
        invoice_count=("invoice_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

print("SUBSCRIPTION METRICS")
print("Total subscriptions:", total_subscriptions)
print("Without end date:", without_end_date)
print("With end date:", with_end_date)

print("\nINVOICE METRICS")
print("Total invoices:", total_invoices)
print("Paid invoices:", paid_invoices)
print("Failed invoices:", failed_invoices)
print("Refunded invoices:", refunded_invoices)
print("Open invoices:", open_invoices)

print("\nINVOICE AMOUNT SUMMARY BY STATUS")
display(invoice_amount_by_status.round(2))


SUBSCRIPTION METRICS
Total subscriptions: 4200
Without end date: 2010
With end date: 2190

INVOICE METRICS
Total invoices: 46000
Paid invoices: 40486
Failed invoices: 3210
Refunded invoices: 1360
Open invoices: 944

INVOICE AMOUNT SUMMARY BY STATUS


,status,invoice_count,total_amount,average_amount
0,failed,3210,2438455.0,759.64
1,open,944,704585.0,746.38
2,paid,40486,30079838.0,742.97
3,refunded,1360,938330.0,689.95


In [13]:
# Create a monthly invoice summary for trend analysis

# Convert invoice issue dates to monthly periods
df["invoice_month"] = df["issued_on"].dt.to_period("M").astype(str)

# Group invoices by month and status
monthly_invoice_summary = (
    df.groupby(["invoice_month", "status"])
    .agg(
        invoice_count=("invoice_id", "count"),
        invoice_amount=("amount", "sum")
    )
    .reset_index()
    .sort_values(["invoice_month", "status"])
)

# Display the most recent 12 months available in the dataset
latest_months = sorted(df["invoice_month"].unique())[-12:]

display(
    monthly_invoice_summary[
        monthly_invoice_summary["invoice_month"].isin(latest_months)
    ]
)


,invoice_month,status,invoice_count,invoice_amount
89,2025-01,failed,105,68995.0
90,2025-01,open,48,36781.0
91,2025-01,paid,1557,1129554.0
92,2025-01,refunded,40,27855.0
93,2025-02,failed,115,102629.0
94,2025-02,open,47,35228.0
95,2025-02,paid,1549,1254770.0
96,2025-02,refunded,48,37471.0
97,2025-03,failed,151,178251.0
98,2025-03,open,44,46138.0


In [14]:
# 1. Monthly invoice summary for the Executive Overview dashboard
monthly_summary = (
    df.groupby(["invoice_month", "status"])
    .agg(
        invoice_count=("invoice_id", "count"),
        invoice_amount=("amount", "sum")
    )
    .reset_index()
)

# 2. Subscription summary by plan
subscription_details = (
    subscriptions_df
    .merge(
        plans_df,
        on="plan_id",
        how="left"
    )
)

plan_summary = (
    subscription_details
    .groupby(["plan_name", "tier"], dropna=False)
    .agg(
        subscription_count=("subscription_id", "count"),
        organization_count=("organization_id", "nunique"),
        average_seats=("seats", "mean")
    )
    .reset_index()
    .sort_values("subscription_count", ascending=False)
)

# 3. Product usage summary by event type
usage_summary = (
    usage_events_df
    .groupby("event_type", dropna=False)
    .agg(
        event_count=("event_id", "count"),
        total_quantity=("quantity", "sum"),
        organization_count=("organization_id", "nunique"),
        user_count=("user_id", "nunique")
    )
    .reset_index()
    .sort_values("event_count", ascending=False)
)

# Display the three summaries for validation
print("MONTHLY INVOICE SUMMARY")
display(monthly_summary.head())

print("SUBSCRIPTION SUMMARY BY PLAN")
display(plan_summary)

print("PRODUCT USAGE SUMMARY")
display(usage_summary)


MONTHLY INVOICE SUMMARY


,invoice_month,status,invoice_count,invoice_amount
0,2023-01,paid,8,1662.0
1,2023-02,paid,18,35123.0
2,2023-03,failed,1,1584.0
3,2023-03,open,1,0.0
4,2023-03,paid,31,25990.0


SUBSCRIPTION SUMMARY BY PLAN


,plan_name,tier,subscription_count,organization_count,average_seats
2,Enterprise,enterprise,571,546,13.523643
1,Business Monthly,business,536,523,13.389925
4,Starter Annual,starter,530,515,13.967925
3,Free,free,523,504,12.933078
7,Team Monthly,team,516,496,13.726744
6,Team Annual,team,514,494,13.873541
5,Starter Monthly,starter,507,494,12.658777
0,Business Annual,business,503,487,14.055666


PRODUCT USAGE SUMMARY


,event_type,event_count,total_quantity,organization_count,user_count
6,report_run,50593,210613,2988,20332
2,dashboard_view,43236,178277,2976,18954
3,export,25345,105427,2882,14237
1,api_call,23248,96350,2843,13445
5,invite_sent,14287,58985,2583,9527
4,integration_sync,12535,52099,2471,8673
0,alert_fired,10756,44010,2386,7550


In [15]:
# Step 1: Import required libraries
from getpass import getpass
from sqlalchemy import create_engine, URL, text

# Step 2: PostgreSQL credentials
pg_user = "postgres"
pg_password = getpass("PostgreSQL password: ")  
pg_host = "localhost"
pg_port = 5432
pg_database = "saas_analytics"

# Step 3: Create the database connection URL
connection_url = URL.create(
    drivername="postgresql+psycopg",
    username=pg_user,
    password=pg_password,
    host=pg_host,
    port=pg_port,
    database=pg_database
)

# Step 4: Create the connection engine
engine = create_engine(connection_url)

# Step 5: Test the connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT current_user, current_database()"))
        row = result.fetchone()

        print("PostgreSQL connected successfully!")
        print("Username:", row[0])
        print("Database:", row[1])

except Exception as error:
    print("Connection failed. Please check your credentials and PostgreSQL server.")
    print("Error:", error)


PostgreSQL password:  ········


PostgreSQL connected successfully!
Username: postgres
Database: saas_analytics


In [18]:
# Load the required libraries
import duckdb
import pandas as pd

# Read all six Parquet files into Pandas DataFrames
organizations_df = duckdb.query(
    "SELECT * FROM read_parquet('../dataset/organizations.parquet')"
).df()

users_df = duckdb.query(
    "SELECT * FROM read_parquet('../dataset/users.parquet')"
).df()

plans_df = duckdb.query(
    "SELECT * FROM read_parquet('../dataset/plans.parquet')"
).df()

subscriptions_df = duckdb.query(
    "SELECT * FROM read_parquet('../dataset/subscriptions.parquet')"
).df()

invoices_df = duckdb.query(
    "SELECT * FROM read_parquet('../dataset/invoices.parquet')"
).df()

usage_events_df = duckdb.query(
    "SELECT * FROM read_parquet('../dataset/usage_events.parquet')"
).df()

# Confirm that all six datasets loaded correctly
datasets = {
    "organizations": organizations_df,
    "users": users_df,
    "plans": plans_df,
    "subscriptions": subscriptions_df,
    "invoices": invoices_df,
    "usage_events": usage_events_df,
}

for table_name, data in datasets.items():
    print(f"{table_name}: {data.shape[0]:,} rows, {data.shape[1]} columns")


organizations: 3,000 rows, 7 columns
users: 26,000 rows, 6 columns
plans: 8 rows, 7 columns
subscriptions: 4,200 rows, 7 columns
invoices: 46,000 rows, 8 columns
usage_events: 180,000 rows, 6 columns


In [17]:
# Save all six DataFrames into PostgreSQL
# if_exists="replace" means a table with the same name will be replaced.
# Use this on a fresh project database or after confirming replacement is safe.

for table_name, data in datasets.items():
    data.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000,
        method="multi"
    )
    print(f"{table_name}: uploaded {len(data):,} rows")


organizations: uploaded 3,000 rows
users: uploaded 26,000 rows
plans: uploaded 8 rows
subscriptions: uploaded 4,200 rows
invoices: uploaded 46,000 rows
usage_events: uploaded 180,000 rows


In [20]:
# Verify that all six tables exist in PostgreSQL
from sqlalchemy import text

with engine.connect() as connection:
    for table_name in datasets.keys():
        query = text(f'SELECT COUNT(*) FROM "{table_name}"')
        row_count = connection.execute(query).scalar()

        print(f"{table_name}: {row_count:,} rows")


organizations: 3,000 rows
users: 26,000 rows
plans: 8 rows
subscriptions: 4,200 rows
invoices: 46,000 rows
usage_events: 180,000 rows


## SQL Business Analysis

### Objective
Analyze SaaS subscriptions, customer organizations, invoices, and product usage to understand business performance.

### Key Business Questions
1. How many organizations are using each subscription plan?
2. What is the total amount of paid, failed, open, and refunded invoices?
3. How does invoice performance change month by month?
4. Which subscription plans are most popular?
5. Which product features are used most frequently?
6. What are the common reasons for subscription cancellations?

### Tools Used
- Python and Pandas
- DuckDB for reading Parquet files
- PostgreSQL for storing data and SQL analysis
- Power BI for dashboard visualization


In [21]:
# Analyze the number of invoices and their total amount by status

query = """
SELECT
    status,
    COUNT(*) AS invoice_count,
    ROUND(SUM(amount)::numeric, 2) AS total_amount,
    ROUND(AVG(amount)::numeric, 2) AS average_amount
FROM invoices
GROUP BY status
ORDER BY invoice_count DESC;
"""

with engine.connect() as connection:
    invoice_status_summary = pd.read_sql(
        text(query), connection
    )

display(invoice_status_summary)


,status,invoice_count,total_amount,average_amount
0,paid,40486,30079838.0,742.97
1,failed,3210,2438455.0,759.64
2,refunded,1360,938330.0,689.95
3,open,944,704585.0,746.38


In [22]:
# Analyze monthly invoice performance by status

query = """
SELECT
    DATE_TRUNC('month', issued_on)::date AS invoice_month,
    status,
    COUNT(*) AS invoice_count,
    ROUND(SUM(amount)::numeric, 2) AS total_amount
FROM invoices
GROUP BY DATE_TRUNC('month', issued_on)::date, status
ORDER BY invoice_month, status;
"""

with engine.connect() as connection:
    monthly_invoice_summary = pd.read_sql(
        text(query), connection
    )

display(monthly_invoice_summary.tail(20))


,invoice_month,status,invoice_count,total_amount
117,2025-08-01,failed,201,161183.0
118,2025-08-01,open,76,43263.0
119,2025-08-01,paid,2608,1854790.0
120,2025-08-01,refunded,98,74095.0
121,2025-09-01,failed,225,154383.0
122,2025-09-01,open,45,27959.0
123,2025-09-01,paid,2659,1884661.0
124,2025-09-01,refunded,84,49931.0
125,2025-10-01,failed,223,146863.0
126,2025-10-01,open,67,39269.0


In [23]:
# Keep only paid invoices and show monthly performance
paid_monthly = monthly_invoice_summary[
    monthly_invoice_summary["status"] == "paid"
].copy()

# Display months in chronological order
paid_monthly = paid_monthly.sort_values("invoice_month")

display(paid_monthly[["invoice_month", "invoice_count", "total_amount"]])


,invoice_month,invoice_count,total_amount
0,2023-01-01,8,1662.0
1,2023-02-01,18,35123.0
4,2023-03-01,31,25990.0
7,2023-04-01,66,92626.0
11,2023-05-01,105,79248.0
15,2023-06-01,135,120288.0
19,2023-07-01,178,175177.0
23,2023-08-01,239,211776.0
27,2023-09-01,306,280396.0
31,2023-10-01,354,312296.0


In [24]:
# Analyze subscription plans and product usage in PostgreSQL

plan_query = """
SELECT
    p.plan_name,
    p.tier,
    COUNT(s.subscription_id) AS subscription_count,
    COUNT(DISTINCT s.organization_id) AS organization_count,
    ROUND(AVG(s.seats)::numeric, 2) AS average_seats
FROM plans p
LEFT JOIN subscriptions s
    ON p.plan_id = s.plan_id
GROUP BY p.plan_name, p.tier
ORDER BY subscription_count DESC;
"""

usage_query = """
SELECT
    event_type,
    COUNT(*) AS event_count,
    COALESCE(SUM(quantity), 0) AS total_quantity,
    COUNT(DISTINCT organization_id) AS organization_count
FROM usage_events
GROUP BY event_type
ORDER BY event_count DESC;
"""

with engine.connect() as connection:
    plan_summary_sql = pd.read_sql(text(plan_query), connection)
    usage_summary_sql = pd.read_sql(text(usage_query), connection)

print("SUBSCRIPTION PLAN ANALYSIS")
display(plan_summary_sql)

print("PRODUCT USAGE ANALYSIS")
display(usage_summary_sql)


SUBSCRIPTION PLAN ANALYSIS


,plan_name,tier,subscription_count,organization_count,average_seats
0,Enterprise,enterprise,571,546,13.52
1,Business Monthly,business,536,523,13.39
2,Starter Annual,starter,530,515,13.97
3,Free,free,523,504,12.93
4,Team Monthly,team,516,496,13.73
5,Team Annual,team,514,494,13.87
6,Starter Monthly,starter,507,494,12.66
7,Business Annual,business,503,487,14.06


PRODUCT USAGE ANALYSIS


,event_type,event_count,total_quantity,organization_count
0,report_run,50593,210613,2988
1,dashboard_view,43236,178277,2976
2,export,25345,105427,2882
3,api_call,23248,96350,2843
4,invite_sent,14287,58985,2583
5,integration_sync,12535,52099,2471
6,alert_fired,10756,44010,2386
